# FraudMind Version 2 Demo

Six specialists. Parallel execution. Conditional routing.

All six agents run concurrently via LangGraph Send fan-out. Ring Detection always runs.
The routing tier controls which specialists are invoked beyond ring.

**Graph flow:** `START → set_routing_tier → [parallel specialists] → council_join → END`

| Case | Scenario | Tier | Expected |
|---|---|---|---|
| 1 | Known fraud ring member: all signals present, ring signature match | full_council (forced) | Ring: block, others: block/escalate |
| 2 | Clean user: home geo, known device, normal payment, no promo abuse | standard | ATO: allow, Payment: allow, Ring: allow |
| 3 | Bot credential stuffing: payload signals only | fast | Ring: allow, Payload: block |

In [ ]:
import sys
sys.path.insert(0, '..')

from src.graph.fraud_graph import fraud_graph
from src.schemas.ato_schemas import MockATOSignals
from src.schemas.payment_schemas import PaymentSignals
from src.schemas.identity_schemas import IdentitySignals
from src.schemas.promo_schemas import PromoSignals
from src.schemas.ring_schemas import RingSignals
from src.schemas.payload_schemas import PayloadSignals

In [ ]:
def print_result(label: str, result: dict) -> None:
    """Pretty-print the output of a graph invocation."""
    print(f"{'='*65}")
    print(f"  {label}")
    print(f"  Routing tier: {result.get('routing_tier', 'unknown')}")
    print(f"{'='*65}")

    verdict_keys = [
        ('ato_verdict',      'ATO'),
        ('payment_verdict',  'PAYMENT'),
        ('identity_verdict', 'IDENTITY'),
        ('promo_verdict',    'PROMO'),
        ('ring_verdict',     'RING'),
        ('payload_verdict',  'PAYLOAD'),
    ]

    for key, label_short in verdict_keys:
        v = result.get(key)
        if v is None:
            continue
        print(f"\n{label_short} VERDICT")
        print(f"  verdict:    {v.verdict.upper()}")
        print(f"  confidence: {v.confidence:.0%}")
        print(f"  signals:    {', '.join(v.primary_signals)}")
        print(f"  reasoning:  {v.reasoning}")
        print(f"  action:     {v.recommended_action}")

    unavailable = result.get('agents_unavailable', [])
    if unavailable:
        print(f"\n  Agents unavailable: {', '.join(unavailable)}")

    print()

## Case 1: Known Ring Member

All six signal sets provided. Ring signature match forces **full_council** regardless of `routing_tier`.

Expected: Ring **block**, others converging on block or escalate.

In [ ]:
case1_state = {
    "primary_entity_id": "acc_ring_001",
    "primary_entity_type": "user",
    "event_type": "login",
    "routing_tier": "standard",  # will be overridden to full_council by ring signature match

    "ato_signals": MockATOSignals(
        account_id="acc_ring_001",
        account_age_days=14,
        is_new_device=True,
        device_id="dev_shared_factory_01",
        login_geo="Lagos, Nigeria",
        account_home_geo="San Jose, CA",
        geo_mismatch=True,
        time_since_last_login_days=1,
        immediate_high_value_action=True,
        email_change_attempted=True,
        password_change_attempted=False,
        support_ticket_language_match=False,
    ),
    "payment_signals": PaymentSignals(
        transaction_id="txn_ring_001",
        account_id="acc_ring_001",
        amount_usd=1800.00,
        merchant_category="electronics",
        merchant_country="Nigeria",
        account_home_country="United States",
        is_international=True,
        transactions_last_1h=7,
        transactions_last_24h=12,
        avg_transaction_amount_30d=75.00,
        amount_vs_avg_ratio=24.0,
        is_first_transaction=False,
        card_present=False,
        days_since_account_creation=14,
    ),
    "identity_signals": IdentitySignals(
        account_id="acc_ring_001",
        account_age_days=14,
        email_domain="tempmail.io",
        is_disposable_email=True,
        phone_country_matches_ip_country=False,
        address_provided=True,
        address_is_commercial=True,
        document_verification_passed=False,
        pii_seen_on_other_accounts=6,
        accounts_created_from_same_device=22,
        accounts_created_from_same_ip=15,
        signup_velocity_same_ip_24h=8,
        name_dob_mismatch=True,
    ),
    "promo_signals": PromoSignals(
        account_id="acc_ring_001",
        promo_code="REF-RING-001",
        promo_type="referral",
        account_age_days=14,
        is_first_order=True,
        promo_uses_this_account=3,
        promo_uses_same_code=1,
        accounts_linked_same_device=10,
        accounts_linked_same_ip=12,
        referrer_account_id="acc_ring_ref_001",
        referrer_account_age_days=7,
        device_shared_with_referrer=True,
        email_domain_pattern_suspicious=True,
        order_cancelled_after_promo=True,
        payout_requested_immediately=True,
    ),
    "ring_signals": RingSignals(
        primary_entity_id="acc_ring_001",
        primary_entity_type="user",
        device_id="dev_shared_factory_01",
        ip_address_hash="hash_known_ring_ip_001",
        accounts_sharing_device=22,
        accounts_sharing_ip=15,
        linked_account_ids=["acc_ring_002", "acc_ring_003", "acc_ring_004"],
        linked_accounts_with_block_verdict=3,
        linked_accounts_with_fraud_confirmed=2,
        shared_payment_method_across_accounts=True,
        payment_method_account_count=7,
        known_ring_signature_match=True,
        ring_signature_id="ring_WEST_AFRICA_001",
        velocity_account_creation_same_ip_7d=18,
        transaction_graph_min_hops_to_fraud=1,
    ),
    "payload_signals": PayloadSignals(
        session_id="sess_ring_001",
        account_id="acc_ring_001",
        user_agent_string="HeadlessChrome/118.0",
        user_agent_is_headless=True,
        tls_fingerprint_ja3="a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4",
        tls_fingerprint_known_bot=True,
        http_header_order_anomaly=True,
        accept_language_missing=True,
        mouse_movement_entropy=0.0,
        keystroke_dynamics_score=0.04,
        requests_per_minute=210.0,
        request_timing_variance_ms=2.1,
        captcha_solve_time_ms=85,
        captcha_solve_pattern_automated=True,
        credential_stuffing_ip_on_blocklist=True,
        login_attempt_count_this_session=9,
        api_endpoint_sequence_anomaly=True,
    ),
}

result1 = fraud_graph.invoke(case1_state)
print_result("Case 1: Known Ring Member (forced full_council)", result1)

## Case 2: Clean User

ATO + Payment + Ring signals. `routing_tier = standard`.

Expected: ATO **allow**, Payment **allow**, Ring **allow**.

In [ ]:
case2_state = {
    "primary_entity_id": "acc_clean_002",
    "primary_entity_type": "user",
    "event_type": "payment",
    "routing_tier": "standard",

    "ato_signals": MockATOSignals(
        account_id="acc_clean_002",
        account_age_days=847,
        is_new_device=False,
        device_id="dev_trusted_047",
        login_geo="San Jose, CA",
        account_home_geo="San Jose, CA",
        geo_mismatch=False,
        time_since_last_login_days=1,
        immediate_high_value_action=False,
        email_change_attempted=False,
        password_change_attempted=False,
        support_ticket_language_match=True,
    ),
    "payment_signals": PaymentSignals(
        transaction_id="txn_clean_002",
        account_id="acc_clean_002",
        amount_usd=94.50,
        merchant_category="grocery",
        merchant_country="United States",
        account_home_country="United States",
        is_international=False,
        transactions_last_1h=1,
        transactions_last_24h=3,
        avg_transaction_amount_30d=88.00,
        amount_vs_avg_ratio=1.07,
        is_first_transaction=False,
        card_present=True,
        days_since_account_creation=847,
    ),
    "ring_signals": RingSignals(
        primary_entity_id="acc_clean_002",
        primary_entity_type="user",
        device_id="dev_trusted_047",
        ip_address_hash="hash_home_ip_002",
        accounts_sharing_device=1,
        accounts_sharing_ip=1,
        linked_account_ids=[],
        linked_accounts_with_block_verdict=0,
        linked_accounts_with_fraud_confirmed=0,
        shared_payment_method_across_accounts=False,
        payment_method_account_count=1,
        known_ring_signature_match=False,
        ring_signature_id=None,
        velocity_account_creation_same_ip_7d=1,
        transaction_graph_min_hops_to_fraud=99,
    ),
}

result2 = fraud_graph.invoke(case2_state)
print_result("Case 2: Clean User (standard tier)", result2)

## Case 3: Bot Credential Stuffing Attack

Payload + Ring signals only. `routing_tier = fast` (ring only via routing, but payload included since it's in state).

Expected: Payload **block**, Ring **allow** (isolated entity).

In [ ]:
case3_state = {
    "primary_entity_id": "acc_target_003",
    "primary_entity_type": "user",
    "event_type": "login",
    "routing_tier": "full_council",  # payload only runs on full_council

    "ring_signals": RingSignals(
        primary_entity_id="acc_target_003",
        primary_entity_type="user",
        device_id="dev_bot_003",
        ip_address_hash="hash_bot_ip_003",
        accounts_sharing_device=2,
        accounts_sharing_ip=3,
        linked_account_ids=[],
        linked_accounts_with_block_verdict=0,
        linked_accounts_with_fraud_confirmed=0,
        shared_payment_method_across_accounts=False,
        payment_method_account_count=1,
        known_ring_signature_match=False,
        ring_signature_id=None,
        velocity_account_creation_same_ip_7d=2,
        transaction_graph_min_hops_to_fraud=99,
    ),
    "payload_signals": PayloadSignals(
        session_id="sess_bot_003",
        account_id="acc_target_003",
        user_agent_string="HeadlessChrome/118.0",
        user_agent_is_headless=True,
        tls_fingerprint_ja3="a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4",
        tls_fingerprint_known_bot=True,
        http_header_order_anomaly=True,
        accept_language_missing=True,
        mouse_movement_entropy=0.0,
        keystroke_dynamics_score=0.02,
        requests_per_minute=195.0,
        request_timing_variance_ms=2.8,
        captcha_solve_time_ms=95,
        captcha_solve_pattern_automated=True,
        credential_stuffing_ip_on_blocklist=True,
        login_attempt_count_this_session=15,
        api_endpoint_sequence_anomaly=True,
    ),
}

result3 = fraud_graph.invoke(case3_state)
print_result("Case 3: Bot Credential Stuffing (full_council, payload + ring only)", result3)